# DMEPOS by Referring Provider & Service labeling

In this workbook we label the cleaned, combined DMEPOS by referring provider and service dataset (`dmepos_rfrhpr.csv`). This workbook produces the key artifact `dmepos_rfrhpr_labeled.csv` which is a critical input for further data modeling and machine learning.

## WARNING! The following notebooks must be run before this one:
* SpIn_1_Artifacts/dmepos_rfrhpr_silver
* SpIn_1_Artifacts/oig_leie_silver.ipynb

## Import and label data

In [1]:
import pandas as pd
# read-in clean rfrhpr data
df = pd.read_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrhpr.csv',dtype={'rfrg_prvdr_state_fips':str,'rfrg_prvdr_zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str
df.head()

,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,rfrg_prvdr_st2,rfrg_prvdr_city,rfrg_prvdr_state_abrvtn,...,tot_suplr_srvcs,avg_suplr_sbmtd_chrg,avg_suplr_mdcr_alowd_amt,avg_suplr_mdcr_pymt_amt,avg_suplr_mdcr_stdzd_amt,year,aapc_desc,tot_suplr_mdcr_pymt_amt,tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr,avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,16,46.336250,20.097500,14.857500,15.280000,2021,Oxygen Delivery Systems and Related Supplies,237.72,-0.425926,0.955638
1,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,19,360.770000,98.223158,72.843684,79.753158,2021,Accessories for Oxygen Delivery Devices,1384.03,-0.347468,0.942913
2,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,11,92.000000,39.230000,31.385455,33.552727,2021,"Wheelchairs, Components, and Accessories",345.24,-0.390508,0.985031
3,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,11,20.000000,10.210909,8.169091,8.456364,2021,"Wheelchairs, Components, and Accessories",89.86,-0.409305,1.022396
4,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,13,272.003846,80.513846,64.407692,84.701538,2021,Accessories for Oxygen Delivery Devices,837.30,-0.419031,0.772909


In [2]:
# read-in clean leie data
labs = pd.read_csv('/dsa/groups/casestudycf25/team02/silver/leie_with_valid_npi_clean.csv',dtype={'npi':int}) # make sure npi numbers are read as int
labs.head()

,general,specialty,npi,excltype,excldate,num_exclusions_alltime,num_exclusion_types_alltime,list_exclusion_types_alltime,num_addresses_alltime,new_id,fraud_flag,excl_1128a1,excl_1128a2,excl_1128a3,excl_1128b5,excl_1128b6,excl_1128b7,excl_brch cia
0,IND- LIC HC SERV PRO,DENTIST,1861529091,1128a3,2024-03-20,1.0,1.0,['1128a3'],1.0,AAMIR-WAHAB-nan-1979-11-16,1,0,0,1,0,0,0,0
1,IND- LIC HC SERV PRO,CHIROPRACTIC,1366544512,1128b7,2021-08-16,1.0,1.0,['1128b7'],1.0,AARON-OXENRIDER-nan-1976-06-07,1,0,0,0,0,0,1,0
2,BUS OWNER/EXEC,LAB - CLINICAL,1306284369,1128a1,2025-11-20,1.0,1.0,['1128a1'],1.0,AARON-ROSSI-nan-1983-02-20,1,1,0,0,0,0,0,0
3,IND- LIC HC SERV PRO,HEALTH CARE AIDE,1376214585,1128a1,2025-01-20,1.0,1.0,['1128a1'],1.0,AARON-WALTON-nan-1971-11-19,1,1,0,0,0,0,0,0
4,BUS OWNER/EXEC,COUNSELING CENTER,1326353459,1128a1,2022-03-20,1.0,1.0,['1128a1'],1.0,AARON-WILLIAMS-nan-1962-10-20,1,1,0,0,0,0,0,0


In [3]:
labs.excldate.dtype

dtype('O')

In [4]:
# return excldate first four digits as int
# print(labs['excldate'][0][:4])

labs['excldate'] = labs.excldate.apply(lambda x: int(x[:4])) # this is different in Hellbender b/c the dates there are "m/d/yyyy" format there

# reduce labs to strictly necessary cols
labs = labs[['npi','fraud_flag','excldate']]

labs.head()

,npi,fraud_flag,excldate
0,1861529091,1,2024
1,1366544512,1,2021
2,1306284369,1,2025
3,1376214585,1,2025
4,1326353459,1,2022


In [5]:
# merge labs w/ df on npi, retain all NPIs in df
df = pd.merge(labs, df, how='right', left_on='npi', right_on='rfrg_npi')
df.head()

,npi,fraud_flag,excldate,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,...,tot_suplr_srvcs,avg_suplr_sbmtd_chrg,avg_suplr_mdcr_alowd_amt,avg_suplr_mdcr_pymt_amt,avg_suplr_mdcr_stdzd_amt,year,aapc_desc,tot_suplr_mdcr_pymt_amt,tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr,avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat
0,NaN,NaN,NaN,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,16,46.336250,20.097500,14.857500,15.280000,2021,Oxygen Delivery Systems and Related Supplies,237.72,-0.425926,0.955638
1,NaN,NaN,NaN,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,19,360.770000,98.223158,72.843684,79.753158,2021,Accessories for Oxygen Delivery Devices,1384.03,-0.347468,0.942913
2,NaN,NaN,NaN,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,11,92.000000,39.230000,31.385455,33.552727,2021,"Wheelchairs, Components, and Accessories",345.24,-0.390508,0.985031
3,NaN,NaN,NaN,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,11,20.000000,10.210909,8.169091,8.456364,2021,"Wheelchairs, Components, and Accessories",89.86,-0.409305,1.022396
4,NaN,NaN,NaN,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,...,13,272.003846,80.513846,64.407692,84.701538,2021,Accessories for Oxygen Delivery Devices,837.30,-0.419031,0.772909


In [6]:
# impute missing values
df['npi'] = df['npi'].fillna(df['rfrg_npi'].astype(int))
df['fraud_flag'] = df['fraud_flag'].fillna(0)
df['excldate'] = df['excldate'].fillna(0)
df['npi'] = df.npi.astype(int)

df.head()

,npi,fraud_flag,excldate,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,...,tot_suplr_srvcs,avg_suplr_sbmtd_chrg,avg_suplr_mdcr_alowd_amt,avg_suplr_mdcr_pymt_amt,avg_suplr_mdcr_stdzd_amt,year,aapc_desc,tot_suplr_mdcr_pymt_amt,tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr,avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat
0,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,16,46.336250,20.097500,14.857500,15.280000,2021,Oxygen Delivery Systems and Related Supplies,237.72,-0.425926,0.955638
1,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,19,360.770000,98.223158,72.843684,79.753158,2021,Accessories for Oxygen Delivery Devices,1384.03,-0.347468,0.942913
2,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,11,92.000000,39.230000,31.385455,33.552727,2021,"Wheelchairs, Components, and Accessories",345.24,-0.390508,0.985031
3,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,11,20.000000,10.210909,8.169091,8.456364,2021,"Wheelchairs, Components, and Accessories",89.86,-0.409305,1.022396
4,1003000480,0.0,0.0,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,...,13,272.003846,80.513846,64.407692,84.701538,2021,Accessories for Oxygen Delivery Devices,837.30,-0.419031,0.772909


In [7]:
def make_labels(fraud_flag, start_exc, data_yr):
    if start_exc >= data_yr: # some fraudulent providers have claims during the year the were added to the LEIE
        return fraud_flag
    else:
        return 0

df['target'] = df[['fraud_flag','excldate','year']].apply(lambda x: make_labels(*x), axis=1)
df.head()

,npi,fraud_flag,excldate,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,...,avg_suplr_sbmtd_chrg,avg_suplr_mdcr_alowd_amt,avg_suplr_mdcr_pymt_amt,avg_suplr_mdcr_stdzd_amt,year,aapc_desc,tot_suplr_mdcr_pymt_amt,tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr,avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat,target
0,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,46.336250,20.097500,14.857500,15.280000,2021,Oxygen Delivery Systems and Related Supplies,237.72,-0.425926,0.955638,0.0
1,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,360.770000,98.223158,72.843684,79.753158,2021,Accessories for Oxygen Delivery Devices,1384.03,-0.347468,0.942913,0.0
2,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,92.000000,39.230000,31.385455,33.552727,2021,"Wheelchairs, Components, and Accessories",345.24,-0.390508,0.985031,0.0
3,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,20.000000,10.210909,8.169091,8.456364,2021,"Wheelchairs, Components, and Accessories",89.86,-0.409305,1.022396,0.0
4,1003000480,0.0,0.0,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,...,272.003846,80.513846,64.407692,84.701538,2021,Accessories for Oxygen Delivery Devices,837.30,-0.419031,0.772909,0.0


### Write-out labeled dataset

In [8]:
# convert target to int and drop unnecessary cols
df = df.drop(['fraud_flag','excldate','rfrg_npi'], axis=1)
df['target'] = df.target.astype(int)

# save as csv for future reference
df.to_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrhpr_labeled.csv', index=False)

df.head()

,npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,rfrg_prvdr_st2,rfrg_prvdr_city,rfrg_prvdr_state_abrvtn,...,avg_suplr_sbmtd_chrg,avg_suplr_mdcr_alowd_amt,avg_suplr_mdcr_pymt_amt,avg_suplr_mdcr_stdzd_amt,year,aapc_desc,tot_suplr_mdcr_pymt_amt,tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr,avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat,target
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,46.336250,20.097500,14.857500,15.280000,2021,Oxygen Delivery Systems and Related Supplies,237.72,-0.425926,0.955638,0
1,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,360.770000,98.223158,72.843684,79.753158,2021,Accessories for Oxygen Delivery Devices,1384.03,-0.347468,0.942913,0
2,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,92.000000,39.230000,31.385455,33.552727,2021,"Wheelchairs, Components, and Accessories",345.24,-0.390508,0.985031,0
3,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,20.000000,10.210909,8.169091,8.456364,2021,"Wheelchairs, Components, and Accessories",89.86,-0.409305,1.022396,0
4,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,272.003846,80.513846,64.407692,84.701538,2021,Accessories for Oxygen Delivery Devices,837.30,-0.419031,0.772909,0
